# Tutorial 03 — CT Segmentation Masks

**What you will learn:**
- What a segmentation mask is and why it is the backbone of medical AI
- How to load and visualize label files alongside CT images
- How to measure organ volumes from a mask
- How label maps differ across body regions

**No GPU needed.** We use pre-generated CT + mask pairs from four body regions.

---
> **Safety reminder:** All images are fully synthetic — generated by AI, not from any real patient.

## What is a segmentation mask?

A **segmentation mask** is a 3D array with the same shape as the CT scan, where each voxel holds an integer:

- **0** = background (nothing of interest)
- **1** = first structure in the anatomy list
- **2** = second structure
- … and so on

This is called a **multi-label map**. It is the standard format used by medical AI segmentation models.

### Why is it valuable?

Creating accurate segmentation masks manually takes a trained radiologist **30–90 minutes per CT scan**. That is why annotated CT datasets are scarce and expensive. Synthetic data generation tools like MAISI can produce unlimited labeled pairs in minutes on a GPU — which is exactly why this technology is transformative for AI research.

In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

PROJECT_ROOT = Path().resolve()
for _ in range(6):
    if (PROJECT_ROOT / 'outputs').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

%matplotlib inline
plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
                     'text.color': 'white', 'axes.labelcolor': '#aaa',
                     'xtick.color': '#aaa', 'ytick.color': '#aaa'})

def apply_window(data, centre=40, width=400):
    lo, hi = centre - width / 2, centre + width / 2
    return np.clip((data - lo) / (hi - lo), 0.0, 1.0)

# Color palette for up to 10 structures
COLORS = ['none', '#ff4d4d', '#4dabf7', '#69db7c', '#ffd43b',
          '#f78c6b', '#da77f2', '#ff8fab', '#63e6be', '#74c0fc', '#ffe066']

print('Setup complete.')

## The four datasets we have

| Dataset | Seed | Structures in the mask |
|---|---|---|
| Chest / Cardio | 71001 | Lung tumor, heart, 5 lung lobes, trachea |
| Abdomen / Tumor | 72001 | Liver, hepatic tumor, spleen, pancreas, stomach, both kidneys, aorta, IVC |
| Pelvis RT | 73001 | Prostate, bladder, left/right femur, left/right hip, sacrum, colon, small bowel |
| Head & Neck | 74001 | Brain, skull, spinal cord, thyroid, trachea, left/right carotid arteries |

In [ ]:
# Dataset registry — label names must match the order in the anatomy_list used during generation
DATASETS = {
    'Chest / Cardio': {
        'image': 'maisi2_showcase_ct_chest_cardio_lung/visuals/sample_001_seed71001/ct_seed71001_image.nii.gz',
        'label': 'maisi2_showcase_ct_chest_cardio_lung/visuals/sample_001_seed71001/ct_seed71001_label.nii.gz',
        'structures': {1:'lung tumor', 2:'heart', 3:'left lung upper',
                       4:'left lung lower', 5:'right lung upper',
                       6:'right lung middle', 7:'right lung lower', 8:'trachea'},
    },
    'Abdomen / Tumor': {
        'image': 'maisi2_showcase_ct_abdomen_organs_tumor/visuals/sample_001_seed72001/ct_seed72001_image.nii.gz',
        'label': 'maisi2_showcase_ct_abdomen_organs_tumor/visuals/sample_001_seed72001/ct_seed72001_label.nii.gz',
        'structures': {1:'liver', 2:'hepatic tumor', 3:'spleen', 4:'pancreas',
                       5:'stomach', 6:'right kidney', 7:'left kidney',
                       8:'aorta', 9:'IVC'},
    },
    'Pelvis RT': {
        'image': 'maisi2_showcase_ct_pelvis_rt/visuals/sample_001_seed73001/ct_seed73001_image.nii.gz',
        'label': 'maisi2_showcase_ct_pelvis_rt/visuals/sample_001_seed73001/ct_seed73001_label.nii.gz',
        'structures': {1:'prostate', 2:'bladder', 3:'left femur', 4:'right femur',
                       5:'left hip', 6:'right hip', 7:'sacrum', 8:'colon', 9:'small bowel'},
    },
    'Head & Neck': {
        'image': 'maisi2_showcase_ct_head_neck/visuals/sample_001_seed74001/ct_seed74001_image.nii.gz',
        'label': 'maisi2_showcase_ct_head_neck/visuals/sample_001_seed74001/ct_seed74001_label.nii.gz',
        'structures': {1:'brain', 2:'skull', 3:'spinal cord', 4:'thyroid',
                       5:'trachea', 6:'L carotid', 7:'R carotid'},
    },
}
print('Datasets registered.')

In [ ]:
# ── CT alone vs CT + mask overlay ─────────────────────────────────────────
#
# The most fundamental visualisation: can you see the structures without a mask?

name = 'Chest / Cardio'
ds = DATASETS[name]
img  = nib.load(str(PROJECT_ROOT / 'outputs' / ds['image'])).get_fdata(dtype=np.float32)
lbl  = nib.load(str(PROJECT_ROOT / 'outputs' / ds['label'])).get_fdata(dtype=np.float32).astype(np.int32)

# Find the slice where the most structures appear
z = int(np.argmax([(lbl[:,:,i] > 0).sum() for i in range(lbl.shape[2])]))

img_slc = np.rot90(img[:, :, z])
lbl_slc = np.rot90(lbl[:, :, z])
windowed = apply_window(img_slc)

cmap = ListedColormap(COLORS[:max(lbl.max(), 1) + 1])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Left: CT only
axes[0].imshow(windowed, cmap='gray', vmin=0, vmax=1, aspect='equal')
axes[0].set_title('CT only\n(soft-tissue window)', fontsize=11)
axes[0].axis('off')

# Right: CT + mask overlay
axes[1].imshow(windowed, cmap='gray', vmin=0, vmax=1, aspect='equal')
masked = np.ma.masked_where(lbl_slc == 0, lbl_slc)
axes[1].imshow(masked, cmap=cmap, vmin=0, vmax=len(COLORS)-1, alpha=0.45, aspect='equal')

# Legend
legend_elements = [Patch(facecolor=COLORS[i], label=ds['structures'][i])
                   for i in ds['structures'] if i <= lbl.max()]
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=8,
               framealpha=0.6, facecolor='#1a1a2e')
axes[1].set_title(f'{name} — CT + structure overlay\n(axial slice z={z})', fontsize=11)
axes[1].axis('off')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/tut03_ct_vs_overlay.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Left: raw CT.  Right: same CT with colour-coded organ masks.')

In [ ]:
# ── Organ volume calculation ───────────────────────────────────────────────
#
# A segmentation mask lets you measure real physical volumes.
# Volume = (number of voxels) × (voxel size in mm³)

print(f'{"Dataset":<20} {"Structure":<22} {"Voxels":>10} {"Volume (mL)":>12}')
print('─' * 68)

for ds_name, ds in DATASETS.items():
    img_nib = nib.load(str(PROJECT_ROOT / 'outputs' / ds['image']))
    lbl_data = nib.load(str(PROJECT_ROOT / 'outputs' / ds['label'])).get_fdata(dtype=np.float32).astype(np.int32)
    sx, sy, sz_ = img_nib.header.get_zooms()[:3]
    voxel_vol_mL = (sx * sy * sz_) / 1000.0  # mm³ → mL

    for label_id, struct_name in ds['structures'].items():
        n_vox = int((lbl_data == label_id).sum())
        if n_vox == 0:
            continue
        vol_mL = n_vox * voxel_vol_mL
        print(f'{ds_name:<20} {struct_name:<22} {n_vox:>10,} {vol_mL:>11.1f} mL')
    print()

print('For reference: a healthy adult liver ≈ 1000–1800 mL, lung ≈ 1500–2500 mL total.')

In [ ]:
# ── All four body regions — centre slice with overlay ─────────────────────

fig, axes = plt.subplots(1, 4, figsize=(18, 6))

for ax, (ds_name, ds) in zip(axes, DATASETS.items()):
    img_d = nib.load(str(PROJECT_ROOT / 'outputs' / ds['image'])).get_fdata(dtype=np.float32)
    lbl_d = nib.load(str(PROJECT_ROOT / 'outputs' / ds['label'])).get_fdata(dtype=np.float32).astype(np.int32)
    n_structs = lbl_d.max()

    # Best slice: most labeled voxels
    z = int(np.argmax([(lbl_d[:,:,i] > 0).sum() for i in range(lbl_d.shape[2])]))
    img_slc = np.rot90(img_d[:, :, z])
    lbl_slc = np.rot90(lbl_d[:, :, z])

    cmap = ListedColormap(COLORS[:n_structs + 1])
    ax.imshow(apply_window(img_slc), cmap='gray', vmin=0, vmax=1, aspect='equal')
    masked = np.ma.masked_where(lbl_slc == 0, lbl_slc)
    ax.imshow(masked, cmap=cmap, vmin=0, vmax=len(COLORS)-1, alpha=0.5, aspect='equal')

    struct_names = list(ds['structures'].values())
    ax.set_title(f'{ds_name}\n{len(struct_names)} structures', fontsize=9)
    ax.axis('off')

fig.suptitle('Four body regions — CT + segmentation mask overlays', fontsize=13)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/tut03_four_regions_overlay.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Each colour represents a different segmented structure.')

In [ ]:
# ── View each structure individually (abdomen example) ────────────────────
#
# This is the view a surgeon or radiation oncologist would use to
# check whether a segmentation is accurate.

ds = DATASETS['Abdomen / Tumor']
img_d = nib.load(str(PROJECT_ROOT / 'outputs' / ds['image'])).get_fdata(dtype=np.float32)
lbl_d = nib.load(str(PROJECT_ROOT / 'outputs' / ds['label'])).get_fdata(dtype=np.float32).astype(np.int32)

structures_to_show = {k: v for k, v in ds['structures'].items() if (lbl_d == k).any()}
n = len(structures_to_show)
cols = min(4, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes_flat = np.array(axes).ravel() if n > 1 else [axes]

for ax, (label_id, struct_name) in zip(axes_flat, structures_to_show.items()):
    # Find the slice with the most voxels for this specific structure
    struct_mask = (lbl_d == label_id)
    best_z = int(np.argmax([struct_mask[:,:,i].sum() for i in range(lbl_d.shape[2])]))
    img_slc = np.rot90(img_d[:, :, best_z])
    struct_slc = np.rot90(struct_mask[:, :, best_z].astype(np.float32))

    ax.imshow(apply_window(img_slc), cmap='gray', vmin=0, vmax=1, aspect='equal')
    highlight = np.zeros((*img_slc.shape, 4), dtype=np.float32)
    c = plt.matplotlib.colors.to_rgba(COLORS[label_id], alpha=0.6)
    highlight[struct_slc > 0] = c
    ax.imshow(highlight, aspect='equal')
    n_vox = struct_mask.sum()
    ax.set_title(f'{struct_name}\n{n_vox:,} voxels  z={best_z}', fontsize=9)
    ax.axis('off')

for ax in axes_flat[len(structures_to_show):]:
    ax.axis('off')

fig.suptitle('Abdomen — each structure shown individually at its best slice', fontsize=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/tut03_individual_structures.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Each panel shows the slice where that structure is largest.')

## Summary

| Concept | Key takeaway |
|---|---|
| Segmentation mask | 3D integer array; each value = structure ID |
| Label 0 | Background (not labeled) |
| Label N | Nth structure in the `anatomy_list` |
| Volume measurement | n_voxels × voxel spacing³ |
| Why it matters | Manual annotation takes 30–90 min/case; synthetic labeled data takes seconds |

### Typical organ volumes (for sanity-checking your synthetic data)

| Organ | Expected range |
|---|---|
| Liver | 1000–1800 mL |
| Spleen | 100–300 mL |
| Kidney (each) | 100–200 mL |
| Heart | 400–900 mL |
| Lung (each) | 1200–2000 mL |
| Prostate | 20–80 mL |
| Bladder (when full) | 300–600 mL |

Checking that your synthetic volumes are in physiologically plausible ranges is a quick first QA step.

## What's next?

**Tutorial 04** — Brain MRI contrasts: T1, T2, FLAIR, SWI, and skull-stripped variants.